In [1]:
import pdfplumber
import pandas as pd
import re
import csv

archivo_pdf = "../data/raw/Metals-ANNEXES-I-A-I-B-II-III-IV.pdf"
archivo_csv = "../data/intermediate/metales_abril.csv"

datos_procesados = []
anexo_actual = "Desconocido"

print("Extrayendo texto línea por línea para no omitir ninguna página...")

with pdfplumber.open(archivo_pdf) as pdf:
    for pagina in pdf.pages:
        texto = pagina.extract_text()
        
        if not texto:
            continue
            
        for linea in texto.split('\n'):
            linea = linea.strip()
            
            if not linea:
                continue
            
            match_anexo = re.search(r'(?i)(Annex\s+[IVX]+(?:-[A-Z])?)', linea)
            if match_anexo:
                anexo_actual = match_anexo.group(1).upper().replace("ANNEX", "Annex")
                continue
                
            encabezados_ignorados = ["Steel", "Description", "Aluminum", "Copper", 
                                     "Steel Derivatives", "Aluminum Derivatives", 
                                     "Copper Articles", "Derivatives"]
            
            if linea in encabezados_ignorados or linea.startswith("Note:") or linea.startswith("If a product") or linea.startswith("The following table"):
                continue
            
            match_codigo = re.match(r'^(\d{4}[\d\.]*)(?:\s+(.*))?$', linea)
            
            if match_codigo:
                codigo = match_codigo.group(1).strip()
                descripcion = match_codigo.group(2).strip() if match_codigo.group(2) else ""
                
                datos_procesados.append({
                    "Anexo": anexo_actual,
                    "Código (HTS)": codigo,
                    "Descripción": descripcion
                })
            else:
                if re.match(r'^\d+$', linea):
                    continue
                    
                if datos_procesados:
                    datos_procesados[-1]["Descripción"] += " " + linea

df = pd.DataFrame(datos_procesados)

# --- MODIFICACIONES ---

# Eliminar todos los registros con Annex IV
df = df[df["Anexo"] != "Annex IV"].copy()

# 1. Eliminar la columna "Descripción"
df = df.drop(columns=["Descripción"])

# 2. Reordenar: primero "Código (HTS)" y después "Anexo"
df = df[["Código (HTS)", "Anexo"]]

# 3. Renombrar columnas a "Code" y "Duty"
df = df.rename(columns={"Código (HTS)": "Code", "Anexo": "Duty"})

# PASO BLINDADO: Quitar espacios/puntos de los extremos y eliminar "2027"
df = df[df["Code"].astype(str).str.strip(" .") != "2027"].copy()

# --- LIMPIEZA DE DUPLICADOS ---

df["Code"] = df["Code"].astype(str)

# Guardar a CSV
df.to_csv(archivo_csv, index=False, encoding='utf-8-sig', quoting=csv.QUOTE_ALL)

print(f"¡Listo! Se extrajeron y limpiaron {len(df)} registros únicos desde la página 1 en adelante.")
print(f"Revisa el nuevo archivo: '{archivo_csv}'")

Extrayendo texto línea por línea para no omitir ninguna página...
¡Listo! Se extrajeron y limpiaron 985 registros únicos desde la página 1 en adelante.
Revisa el nuevo archivo: '../data/intermediate/metales_abril.csv'
